# Capitolo 5 — Cosa ha imparato la rete: convoluzione a mano, filtri, mappe, Grad-CAM (§ 5.2, 5.6, 5.11)
Richiede `cifar10_cnn.ipynb` (per `cnn_cifar10.pt`) e `transfer_gatti_cani.ipynb` (per il modello gatti/cani).

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

from torchvision import models
from PIL import Image
X_te, y_te = dati.cifar10_mirror()[2:]
img = X_te[7].astype(float) / 255; gr = img.mean(2)

## 5.2 La convoluzione, fatta a mano

In [ ]:
def conv2d(immagine, filtro):
    H, W = immagine.shape; k = filtro.shape[0]
    uscita = np.zeros((H - k + 1, W - k + 1))
    for i in range(uscita.shape[0]):
        for j in range(uscita.shape[1]): uscita[i, j] = np.sum(immagine[i:i + k, j:j + k] * filtro)
    return uscita

kv = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], float); kh = kv.T; kmedia = np.ones((3, 3)) / 9; klap = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], float)
fig, ax = plt.subplots(1, 5, figsize=(11, 2.4))
ax[0].imshow(gr, cmap="gray"); ax[0].set_title(dati.CIFAR10_CLASSI[y_te[7]])
for a, k, t in zip(ax[1:], (kv, kh, kmedia, klap), ("bordi verticali", "bordi orizzontali", "media (sfoca)", "laplaciano")):
    a.imshow(conv2d(gr, k), cmap="gray"); a.set_title(t)
for a in ax: a.axis("off")
plt.show()

## 5.6 I filtri appresi e le mappe di caratteristiche della CNN di CIFAR-10

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(2048, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 10))
    def forward(self, x): return self.fc(self.conv(x))
cnn = CNN(); cnn.load_state_dict(torch.load("cnn_cifar10.pt", weights_only=True)); cnn.eval()

W = cnn.conv[0].weight.detach()                     # [32, 3, 3, 3]
fig, ax = plt.subplots(2, 16, figsize=(12, 1.8))
for i, a in enumerate(ax.flat):
    w = W[i].permute(1, 2, 0).numpy(); a.imshow((w - w.min()) / (w.max() - w.min() + 1e-8)); a.axis("off")
plt.suptitle("i 32 filtri del primo strato"); plt.show()

x = torch.from_numpy(X_te[7]).permute(2, 0, 1).float() / 255; x = ((x - 0.5) / 0.5).unsqueeze(0)
with torch.no_grad():
    h1 = torch.relu(cnn.conv[0](x)); h2 = torch.relu(cnn.conv[3](cnn.conv[:3](x))); h3 = torch.relu(cnn.conv[6](cnn.conv[:6](x)))
fig, ax = plt.subplots(3, 8, figsize=(12, 4.5))
for r, (h, t) in enumerate(((h1, "strato 1"), (h2, "strato 2"), (h3, "strato 3"))):
    for c in range(8): ax[r, c].imshow(h[0, c], cmap="viridis"); ax[r, c].axis("off")
    ax[r, 0].set_title(t, loc="left")
plt.show()

## 5.11 Grad-CAM sul classificatore gatti/cani

In [ ]:
import json
classi = json.load(open("../cap09_produzione/gatti_cani.json"))["classi"]
m = models.resnet18(weights=None); m.fc = nn.Linear(512, 2); m.load_state_dict(torch.load("../cap09_produzione/gatti_cani.pt", weights_only=True)); m.eval()
prep = models.ResNet18_Weights.DEFAULT.transforms()
attivazioni, gradienti = {}, {}
m.layer4.register_forward_hook(lambda mod, i, o: attivazioni.__setitem__("a", o))
m.layer4.register_full_backward_hook(lambda mod, gi, go: gradienti.__setitem__("g", go[0]))

def gradcam(percorso):
    im = Image.open(percorso).convert("RGB"); x = prep(im).unsqueeze(0)
    uscita = m(x); k = uscita.argmax().item(); m.zero_grad(); uscita[0, k].backward()
    a, g = attivazioni["a"][0].detach(), gradienti["g"][0]
    cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0)); cam = (cam / cam.max()).numpy()
    w0, h0 = im.size; s = 256 / min(w0, h0); im2 = im.resize((round(w0 * s), round(h0 * s))); l, t = (im2.width - 224) // 2, (im2.height - 224) // 2
    return im2.crop((l, t, l + 224, t + 224)), cam, classi[k], torch.softmax(uscita, 1)[0, k].item()

foto = ["foto/test/gatto/0600.jpg", "foto/test/cane/0601.jpg", "foto/test/gatto/0612.jpg", "foto/test/cane/0630.jpg"]
fig, ax = plt.subplots(2, 4, figsize=(11, 5.5))
for j, f in enumerate(foto):
    im, cam, cl, p = gradcam(f)
    ax[0, j].imshow(im); ax[0, j].set_title(f"{cl} ({p:.0%})"); ax[0, j].axis("off")
    ax[1, j].imshow(im); ax[1, j].imshow(np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224))), cmap="jet", alpha=0.45); ax[1, j].axis("off")
plt.show()